# 第11次课：文件读写

**地学编程基础** | 2学时（90分钟）

---

## 📚 本节课学习目标

完成本节课后，你应该能够：

1. **掌握** `open()` + `with` 语句打开文件；
2. **掌握**三种读取文件的方式：`read()` / `readlines()` / `for line in f`；
3. **掌握**写入文件：`write()` / `writelines()`；
4. **理解**读写模式（`'r'` / `'w'` / `'a'`）；
5. **理解**编码（`encoding='utf-8'`）；
6. **掌握** `csv` 模块的基础用法（`csv.reader`、`csv.writer`、`csv.DictReader`）；
7. **能够**完成'读 CSV → 处理 → 写 CSV'的完整工作流。

---

## 🎯 这节课的特殊意义

**这是模块二最有'工程价值'的一节课**——你将**第一次脱离'代码内置数据'，对接真实文件**。

**前面 10 次课**：所有数据都写在代码里（`elevations = [2530, 1820, ...]`）

**今天起**：从本地文件读数据 + 把结果写到本地文件

**这意味着什么？**——你的程序可以处理**真实的、大量的、来自外部的数据**！

---

## 📂 配套数据文件

本节课的目录结构：

```
lesson11/
├── Lesson11_文件读写.ipynb     ← 你正在看的这个
└── data/
    ├── cities_climate.csv      ← 15 个城市的气候数据
    └── elevations.txt          ← 12 个观测点的海拔
```

请确认你的 ipynb 和 `data/` 目录在同一个文件夹下。

---

## 一、open() —— 打开文件的基础

### 1.1 一个简单的开始

**🔑 三要素**：
- 文件路径
- 模式（读 / 写 / 追加）
- 编码（处理中文必须）

In [ ]:
# 最简单的读取一个文件
f = open("data/elevations.txt", "r", encoding="utf-8")
content = f.read()        # 读取全部内容
f.close()                 # 一定要记得关闭！

print(content)

**关键观察**：
- `open()` 返回一个**文件对象**
- `f.read()` 把整个文件读进来（字符串）
- `f.close()` **必须调用** —— 不然文件可能没保存好/没释放

**❓ 总是怕忘记 close 怎么办？** —— 用 `with` 语句（推荐！）

### 1.2 ⭐ 推荐写法：with 语句

**用 `with` 语句打开文件——会自动关闭**：

In [ ]:
# 推荐写法：用 with 语句
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    content = f.read()
    print(content)
# with 块结束，文件自动关闭——不需要手动 close()

**🌟 with 的好处**：
1. 自动关闭文件 —— 不会忘记
2. 即使中途出错，也能保证关闭
3. 代码更清晰——'**这个 with 块在处理这个文件**'

**📌 经验法则**：**今天起，所有文件操作都用 `with`**——不要再手动 `f = open(...)` + `f.close()`。

### 1.3 三种打开模式

| 模式 | 含义 | 行为 | 文件不存在时 |
|---|---|---|---|
| `'r'` | 读（read）| 只读，**不能写** | 报错 |
| `'w'` | 写（write）| 写入，**会清空原有内容** | 创建新文件 |
| `'a'` | 追加（append）| 写入，**追加到末尾** | 创建新文件 |

**⚠️ 高频踩坑**：
- `'w'` 模式会**清空文件**——别误用！
- 想保留原内容并添加 → 用 `'a'`

### 1.4 ⭐ 编码：encoding='utf-8'

**Windows 系统的中文文件特别容易踩坑**——需要明确指定编码。

**经验法则**：
- **所有文件操作都加 `encoding='utf-8'`** —— 最安全的做法
- 中文 Windows 系统默认编码是 GBK，会和 UTF-8 文件不兼容
- 现代文件几乎都是 UTF-8 编码的

In [ ]:
# ❌ 不加 encoding 在 Windows 中文环境可能报错
# with open("data/elevations.txt") as f:
#     content = f.read()    # 可能 UnicodeDecodeError

# ✅ 永远加上 encoding='utf-8'
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    content = f.read()

---

## 二、读取文件 —— 三种方式

### 2.1 方式一：read() —— 一次读全部

**适合**：小文件、需要把全部内容当作一个字符串处理。

In [ ]:
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    content = f.read()    # content 是一个大字符串

print(type(content))    # <class 'str'>
print(len(content), "个字符")
print("---")
print(content)

### 2.2 方式二：readlines() —— 读成行的列表

**适合**：知道文件不太大，想要按行处理。

In [ ]:
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()    # lines 是一个列表，每个元素是一行（带换行符）

print(type(lines))    # <class 'list'>
print(f"共 {len(lines)} 行")

# 看前 3 行
for line in lines[:3]:
    print(repr(line))    # repr 显示真实内容（包括 \n）

**📌 注意**：`readlines()` 读出来的每一行**都带着 `\n` 换行符**——后面处理时通常需要 `strip()` 去掉。

### 2.3 ⭐ 方式三：for line in f —— 逐行读取（推荐）

**适合**：所有场景——**这是 Python 处理文件的标准模式**。

In [ ]:
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    for line in f:                  # 文件对象本身是可迭代的
        line = line.strip()         # 去掉换行符（标准做法！）
        print(line)

**🌟 这种写法的好处**：
1. 内存友好——大文件也不会爆内存（不像 read 一次读全部）
2. 代码简洁
3. **配合 strip 处理换行符**——这是黄金组合

### 2.4 三种方式对比

| 方式 | 返回类型 | 内存占用 | 适用 |
|---|---|---|---|
| `read()` | 一个大字符串 | 全部加载 | 小文件、整体处理 |
| `readlines()` | 字符串列表 | 全部加载 | 中等文件、行处理 |
| `for line in f` | 一行一行迭代 | 只占一行的内存 | **推荐默认用** |

**经验法则**：默认用 `for line in f` —— 简单且高效。

### 2.5 综合应用：解析 elevations.txt

文件内容（每行：`观测点名 海拔`）：
```
塔克拉玛干 1200
敦煌 1100
...
```

**目标**：读取并解析成 `[(name, elevation), ...]` 的列表。

In [ ]:
# 综合应用：读 + 解析
stations = []

with open("data/elevations.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()        # 1. 去换行符
        if not line:
            continue               # 2. 跳过空行（健壮性）
        parts = line.split()       # 3. 用空格分隔
        name = parts[0]
        elevation = int(parts[1])
        stations.append((name, elevation))

# 验证
for s in stations[:5]:
    print(s)

print(f"\n共解析 {len(stations)} 个观测点")

**🌟 这就是文件读取的完整工作流**——
1. 用 `with` + `open` 打开
2. `for line in f` 逐行读
3. `strip()` 清理
4. `split()` + `int()/float()` 解析
5. `append` 到结果列表

---

## 🎯 即时练习①

**练习目标**：熟练用 with + for line in f 读取文件并处理。

**练习题**：读取 `data/elevations.txt`，完成：

1. **打印每一行**（去掉换行符）；
2. **筛选并打印海拔 > 3000 米的观测点**；
3. **统计**有多少个观测点超过 5000 米；
4. **找出**海拔最高的观测点（提示：先解析成列表 + max + lambda）。

In [ ]:
# ===== 即时练习①.1：打印每一行 =====




In [ ]:
# ===== 即时练习①.2：筛选海拔 > 3000 米 =====




In [ ]:
# ===== 即时练习①.3：统计 > 5000 米的观测点数 =====




In [ ]:
# ===== 即时练习①.4：找出海拔最高的观测点 =====




---

## 三、写入文件

### 3.1 write() —— 写入字符串

In [ ]:
# 把字符串写入新文件（'w' 模式 + 文件不存在则创建）
with open("output_demo.txt", "w", encoding="utf-8") as f:
    f.write("第一行\n")           # 写入字符串
    f.write("第二行\n")
    f.write("第三行：海拔 3650 米\n")

print("写入完成！请打开 output_demo.txt 查看")

# 写入后立即读出来验证
with open("output_demo.txt", "r", encoding="utf-8") as f:
    print(f.read())

**📌 重要细节**：
- `f.write()` **不会自动加换行符**——必须手动写 `\n`
- 这跟 `print()` 不同（print 默认换行）

### 3.2 写入多行：循环 + write

In [ ]:
# 实用场景：把分析结果写入文件
stations = [
    ("塔克拉玛干", 1200),
    ("敦煌", 1100),
    ("拉萨", 3650),
]

with open("output_stations.txt", "w", encoding="utf-8") as f:
    f.write("=== 观测点列表 ===\n")
    for name, elevation in stations:
        line = f"{name}: {elevation} 米\n"
        f.write(line)
    f.write("=== 完毕 ===\n")

# 验证
with open("output_stations.txt", "r", encoding="utf-8") as f:
    print(f.read())

### 3.3 ⚠️ 'w' vs 'a' —— 高频踩坑

In [ ]:
# 'w' 模式会清空文件！
with open("output_test.txt", "w", encoding="utf-8") as f:
    f.write("第一次写入\n")

with open("output_test.txt", "w", encoding="utf-8") as f:    # 又用 'w'
    f.write("第二次写入\n")

# 看结果
with open("output_test.txt", "r", encoding="utf-8") as f:
    print(f.read())    # 只有'第二次写入'——'第一次'被清掉了！

In [ ]:
# 'a' 模式会追加 —— 保留原内容
with open("output_test.txt", "w", encoding="utf-8") as f:
    f.write("原始内容\n")

with open("output_test.txt", "a", encoding="utf-8") as f:    # 追加！
    f.write("追加内容\n")

with open("output_test.txt", "r", encoding="utf-8") as f:
    print(f.read())    # 两行都在！

**🛡️ 防御口诀**：

- **新建/重写文件用 `'w'`**（清空再写）
- **追加内容用 `'a'`**（不动原内容）
- **绝对不要混用** —— 看清楚每次的模式

---

## 🎯 即时练习②

**练习目标**：熟练用 with + write 写入文件。

**练习题**：

用代码生成下面这个'气象报告'文件 `weather_report.txt`：

```
=== 7 月气象统计报告 ===
城市：北京
平均温度：32.5°C
总降雨量：85 mm
------
城市：上海
平均温度：34.8°C
总降雨量：120 mm
------
城市：广州
平均温度：33.2°C
总降雨量：250 mm
------
=== 报告结束 ===
```

**数据**：
```python
data = [
    ("北京", 32.5, 85),
    ("上海", 34.8, 120),
    ("广州", 33.2, 250),
]
```

In [ ]:
# ===== 即时练习② =====
data = [
    ("北京", 32.5, 85),
    ("上海", 34.8, 120),
    ("广州", 33.2, 250),
]




---

## 四、CSV 模块 —— 处理表格数据

**CSV（Comma-Separated Values）**：用逗号分隔的表格文件，地学专业最常见的数据格式之一。

**配套文件 cities_climate.csv 的内容**：
```
city,latitude,longitude,annual_temp,annual_rainfall
哈尔滨,45.75,126.65,5.6,530
北京,39.90,116.40,13.2,580
...
```

### 4.1 ❌ 不要手动 split(',')

**不推荐**手动用 `split(',')` 处理 CSV——因为 CSV 有很多坑：
- 数据中包含逗号：`"北京, 中国",116.40` —— `split(',')` 会拆错
- 数据中包含换行符
- 数据中包含引号

**正确做法**：用 Python 的 `csv` 模块。

### 4.2 csv.reader —— 基础读取

In [ ]:
import csv

with open("data/cities_climate.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

**关键观察**：
- `csv.reader(f)` 包装文件对象
- `for row in reader` —— 每个 `row` 是一个**列表**（每列一个元素）
- **第一行通常是'表头'**（city, latitude, ... ）
- 所有值都是**字符串**——数字也是 `'45.75'`，需要 `float()` 转换

In [ ]:
# 实用：跳过表头 + 转换类型
import csv

cities = []

with open("data/cities_climate.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)        # 跳过第一行（表头）
    print("表头：", header)
    
    for row in reader:
        name = row[0]
        lat = float(row[1])
        lon = float(row[2])
        temp = float(row[3])
        rain = float(row[4])
        cities.append((name, lat, lon, temp, rain))

# 验证
print(f"\n共 {len(cities)} 个城市")
for c in cities[:3]:
    print(c)

### 4.3 ⭐ csv.DictReader —— 更优雅

**DictReader 把每行包装成字典**——用'**列名**'访问，不用记位置！

In [ ]:
import csv

with open("data/cities_climate.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        # row 是字典 {列名: 值}
        print(f"{row['city']}：{row['annual_temp']}°C")

**🌟 DictReader 的好处**：
- **不需要记列号**（不用想'温度是第 4 列还是第 5 列'）
- **代码更可读**（`row['city']` 比 `row[0]` 直观多了）
- **列顺序变了也不影响**

**📌 经验法则**：**默认用 DictReader**——除非有特殊需要才用 reader。

In [ ]:
# 用 DictReader 处理城市数据
import csv

cities = []

with open("data/cities_climate.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        # 数字字段需要转换类型
        cities.append({
            "city": row["city"],
            "lat": float(row["latitude"]),
            "lon": float(row["longitude"]),
            "temp": float(row["annual_temp"]),
            "rain": float(row["annual_rainfall"]),
        })

# 现在 cities 是字典列表 —— 用列名访问任何字段
for c in cities[:3]:
    print(c)

In [ ]:
# 综合应用：找出降水量最高的城市
wettest = max(cities, key=lambda x: x["rain"])
print(f"降水量最高：{wettest['city']}，{wettest['rain']} mm")

# 找出气温 > 20°C 的城市
hot = []
for c in cities:
    if c["temp"] > 20:
        hot.append(c["city"])
print(f"气温 > 20°C 的城市：{hot}")

### 4.4 写 CSV：csv.writer 和 csv.DictWriter

In [ ]:
# csv.writer：基础写法
import csv

data = [
    ["name", "elevation"],         # 表头
    ["塔克拉玛干", 1200],
    ["敦煌", 1100],
    ["拉萨", 3650],
]

# 注意：写 CSV 文件时建议加 newline=''（避免 Windows 多空行）
with open("output_writer.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    
    for row in data:
        writer.writerow(row)
    
    # 也可以一次写多行
    # writer.writerows(data)

# 验证
with open("output_writer.csv", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# csv.DictWriter：用字典写 —— 更优雅
import csv

data = [
    {"name": "塔克拉玛干", "elevation": 1200},
    {"name": "敦煌", "elevation": 1100},
    {"name": "拉萨", "elevation": 3650},
]

with open("output_dictwriter.csv", "w", encoding="utf-8", newline="") as f:
    fieldnames = ["name", "elevation"]    # 必须指定列名
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    writer.writeheader()                  # 写入表头
    writer.writerows(data)                # 一次写多行

# 验证
with open("output_dictwriter.csv", "r", encoding="utf-8") as f:
    print(f.read())

**📌 写 CSV 的两个细节**：
- **`newline=""`** —— 必须加！否则 Windows 上每行之间会有空行
- **`writeheader()`** —— DictWriter 写表头的专用方法

---

## 五、文件路径基础

### 5.1 相对路径 vs 绝对路径

**相对路径**：相对于'当前工作目录'的位置
- `"data/cities.csv"` —— 同目录下的 data 文件夹
- `"./data/cities.csv"` —— 同上（`./` 表示当前目录）
- `"../cities.csv"` —— 上一级目录

**绝对路径**：完整的从根目录开始的路径
- Windows：`"C:/Users/张三/data/cities.csv"`
- Linux/Mac：`"/home/zhang/data/cities.csv"`

**📌 经验法则**：**优先用相对路径**——代码可以分享给别人。

### 5.2 ⚠️ Windows 路径的反斜杠问题

**Windows 路径里用反斜杠 `\`，但反斜杠在 Python 字符串里是转义字符**：

In [ ]:
# ❌ Windows 路径直接复制 —— 可能出错
# path1 = "C:\Users\test\data.csv"      # \U \t 都是转义字符！

# ✅ 三种安全写法：
path1 = "C:/Users/test/data.csv"        # 1. 用正斜杠（最简单）
path2 = r"C:\Users\test\data.csv"       # 2. r"" 原始字符串
path3 = "C:\\Users\\test\\data.csv"     # 3. 双反斜杠（不推荐，丑）

print(path1)
print(path2)
print(path3)

**📌 经验法则**：
- **代码里所有路径都用正斜杠 `/`**——Windows 也能识别
- 这是跨平台的最佳实践

---

## 🎯 即时练习③（综合实战）

**练习目标**：完成'读 CSV → 处理 → 写 CSV'完整工作流。

**情境**：你拿到了 `data/cities_climate.csv`——15 个城市的年度气候数据。

**任务**：

**步骤1**：用 `csv.DictReader` 读取所有数据，存成字典列表 `cities`。

**步骤2**：筛选出**年降水量 > 1000 mm** 的城市（多雨城市）。

**步骤3**：把筛选结果写入新文件 `output_rainy.csv`，格式：

```
city,latitude,annual_rainfall
上海,31.23,1200
广州,23.13,1700
...
```

**步骤4**：再写一个汇总文件 `output_summary.txt`，内容：

```
=== 多雨城市汇总（年降水 > 1000 mm）===
共 X 个城市：
  上海（31.23°N）：1200 mm
  ...
  
平均年降水量：YY mm
降水最多：ZZZ（NNN mm）
```

**提示**：
- 用 `csv.DictReader` 读
- 用循环 + if 筛选
- 用 `csv.DictWriter` 写第一个 CSV
- 用 `with open + write` 写第二个 TXT
- 用 `import statistics` 算平均

In [ ]:
# ===== 即时练习③ 综合实战 =====
import csv
import statistics

# 步骤1：读取所有数据
cities = []




In [ ]:
# 步骤2：筛选多雨城市（年降水 > 1000）
rainy_cities = []




In [ ]:
# 步骤3：写入 output_rainy.csv




In [ ]:
# 步骤4：写入 output_summary.txt




---

## 六、本节课小结

### 知识点回顾

| 知识点 | 关键语法 |
|---|---|
| 推荐打开方式 | `with open(...) as f:` |
| 编码 | `encoding='utf-8'` |
| 模式 | `'r'` 读 / `'w'` 写（清空）/ `'a'` 追加 |
| 读全部 | `f.read()` |
| 读多行 | `f.readlines()` |
| 逐行读（推荐） | `for line in f` |
| 写一行 | `f.write("text\n")` |
| CSV 读 | `csv.reader(f)` / `csv.DictReader(f)` |
| CSV 写 | `csv.writer(f)` / `csv.DictWriter(f)` |
| 写 CSV 的细节 | `newline=''` 必须加 |

### 重点强调（重要程度从高到低）

1. ⭐⭐⭐ **永远用 `with`** —— 不要手动 `open` + `close`
2. ⭐⭐⭐ **永远加 `encoding='utf-8'`** —— 中文文件必备
3. ⭐⭐⭐ **`for line in f` 是首选读取方式** —— 简洁高效
4. ⭐⭐ **`csv.DictReader` 比 `csv.reader` 优雅** —— 用列名访问
5. ⭐⭐ **`'w'` 会清空文件** —— 想保留原内容用 `'a'`
6. ⭐ **`f.write()` 不会自动换行** —— 手动 `\n`
7. ⭐ **写 CSV 加 `newline=''`** —— 避免 Windows 空行

### 课后作业

**1. 【基础】文件读写综合（必做）**

1. 自己创建一个 `data/my_stations.txt`，内容（每行：观测站名 海拔）：
   ```
   珠峰 8848
   K2 8611
   干城章嘉 8586
   ...（自己再加 7 个）
   ```
2. 用代码读取这个文件，解析成 `[(name, elevation), ...]`；
3. 找出最高的 3 个观测点；
4. 把最高的 3 个写入 `output_top3.txt`。

**2. 【进阶】CSV 数据处理（必做）**

用 `data/cities_climate.csv` 完成：
1. 读取数据；
2. 计算每个城市的'**舒适度评分**'（自定义公式：`100 - abs(temp - 20) * 5 - abs(rain - 800) * 0.05`）；
3. 把'城市 + 舒适度评分'写入新 CSV 文件 `comfort_score.csv`，按评分降序排列；
4. 输出'最舒适的城市'和'最不舒适的城市'到屏幕。

**3. 【挑战】合并多个 CSV 文件（必做）**

1. 把 `cities_climate.csv` 拆成两个文件：北方城市（纬度 ≥ 30）和南方城市（纬度 < 30）；
2. 文件名分别为 `output_north.csv` 和 `output_south.csv`；
3. 然后再把这两个文件合并成 `output_merged.csv`，并加一列 `region`（值为 'North' 或 'South'）。

---

下节课预告：**第12次课 异常与调试** —— 学习 `try-except` 优雅处理错误、调试技巧。读文件出错怎么办？这就是下次课的主题之一。

---
---

# 📎 附录：参考答案

> **建议先独立完成所有练习题再查看答案。**

In [ ]:
# ===== 即时练习①.1 参考答案 =====
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

In [ ]:
# ===== 即时练习①.2 参考答案 =====
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        name = parts[0]
        elevation = int(parts[1])
        if elevation > 3000:
            print(f"{name}: {elevation} 米")

In [ ]:
# ===== 即时练习①.3 参考答案 =====
count = 0

with open("data/elevations.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        elevation = int(parts[1])
        if elevation > 5000:
            count = count + 1

print(f"超过 5000 米的观测点：{count} 个")

In [ ]:
# ===== 即时练习①.4 参考答案 =====
stations = []

with open("data/elevations.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        name = parts[0]
        elevation = int(parts[1])
        stations.append((name, elevation))

highest = max(stations, key=lambda x: x[1])
print(f"海拔最高：{highest[0]}（{highest[1]} 米）")

In [ ]:
# ===== 即时练习② 参考答案 =====
data = [
    ("北京", 32.5, 85),
    ("上海", 34.8, 120),
    ("广州", 33.2, 250),
]

with open("weather_report.txt", "w", encoding="utf-8") as f:
    f.write("=== 7 月气象统计报告 ===\n")
    
    for city, temp, rain in data:
        f.write(f"城市：{city}\n")
        f.write(f"平均温度：{temp}°C\n")
        f.write(f"总降雨量：{rain} mm\n")
        f.write("------\n")
    
    f.write("=== 报告结束 ===\n")

# 验证
with open("weather_report.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# ===== 即时练习③ 步骤1 参考答案 =====
import csv

cities = []

with open("data/cities_climate.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        cities.append({
            "city": row["city"],
            "lat": float(row["latitude"]),
            "lon": float(row["longitude"]),
            "temp": float(row["annual_temp"]),
            "rain": float(row["annual_rainfall"]),
        })

print(f"共读取 {len(cities)} 个城市")
for c in cities[:3]:
    print(c)

In [ ]:
# ===== 即时练习③ 步骤2 参考答案 =====
rainy_cities = []

for c in cities:
    if c["rain"] > 1000:
        rainy_cities.append(c)

print(f"多雨城市数量：{len(rainy_cities)}")
for c in rainy_cities:
    print(f"  {c['city']}：{c['rain']} mm")

In [ ]:
# ===== 即时练习③ 步骤3 参考答案 =====
import csv

with open("output_rainy.csv", "w", encoding="utf-8", newline="") as f:
    fieldnames = ["city", "latitude", "annual_rainfall"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    writer.writeheader()
    
    for c in rainy_cities:
        writer.writerow({
            "city": c["city"],
            "latitude": c["lat"],
            "annual_rainfall": c["rain"],
        })

# 验证
print("=== output_rainy.csv 内容 ===")
with open("output_rainy.csv", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# ===== 即时练习③ 步骤4 参考答案 =====
import statistics

rains = [c["rain"] for c in rainy_cities]
avg_rain = statistics.mean(rains)

wettest = max(rainy_cities, key=lambda x: x["rain"])

with open("output_summary.txt", "w", encoding="utf-8") as f:
    f.write("=== 多雨城市汇总（年降水 > 1000 mm）===\n")
    f.write(f"共 {len(rainy_cities)} 个城市：\n")
    
    for c in rainy_cities:
        f.write(f"  {c['city']}（{c['lat']}°N）：{c['rain']:.0f} mm\n")
    
    f.write(f"\n平均年降水量：{avg_rain:.0f} mm\n")
    f.write(f"降水最多：{wettest['city']}（{wettest['rain']:.0f} mm）\n")

# 验证
print("=== output_summary.txt 内容 ===")
with open("output_summary.txt", "r", encoding="utf-8") as f:
    print(f.read())

---

## 课后作业参考答案

In [ ]:
# ===== 课后作业 1：文件读写综合 =====
# 步骤1：先用代码生成 my_stations.txt（也可以手动创建）
stations_data = """珠峰 8848
K2 8611
干城章嘉 8586
洛子峰 8516
马卡鲁 8485
卓奥友 8201
道拉吉里 8167
马纳斯卢 8163
南迦帕尔巴特 8125
安纳普尔纳 8091
"""

import os
os.makedirs("data", exist_ok=True)

with open("data/my_stations.txt", "w", encoding="utf-8") as f:
    f.write(stations_data)

# 步骤2：读取并解析
stations = []
with open("data/my_stations.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        name = parts[0]
        elevation = int(parts[1])
        stations.append((name, elevation))

# 步骤3：找最高的 3 个
top3 = sorted(stations, key=lambda x: x[1], reverse=True)[:3]
print("最高的 3 个：")
for s in top3:
    print(f"  {s[0]}: {s[1]} 米")

# 步骤4：写入 output_top3.txt
with open("output_top3.txt", "w", encoding="utf-8") as f:
    f.write("=== 海拔最高的 3 个观测点 ===\n")
    for name, elev in top3:
        f.write(f"{name}: {elev} 米\n")

print("\nTop 3 已写入 output_top3.txt")

In [ ]:
# ===== 课后作业 2：CSV 数据处理 =====
import csv

# 步骤1：读取数据
cities = []
with open("data/cities_climate.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        cities.append({
            "city": row["city"],
            "temp": float(row["annual_temp"]),
            "rain": float(row["annual_rainfall"]),
        })

# 步骤2：计算舒适度评分
for c in cities:
    score = 100 - abs(c["temp"] - 20) * 5 - abs(c["rain"] - 800) * 0.05
    c["score"] = round(score, 1)

# 步骤3：按评分降序写 CSV
cities_sorted = sorted(cities, key=lambda x: x["score"], reverse=True)

with open("comfort_score.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["city", "score"])
    writer.writeheader()
    for c in cities_sorted:
        writer.writerow({"city": c["city"], "score": c["score"]})

# 步骤4：输出最值
best = cities_sorted[0]
worst = cities_sorted[-1]
print(f"最舒适：{best['city']}（评分 {best['score']}）")
print(f"最不舒适：{worst['city']}（评分 {worst['score']}）")

In [ ]:
# ===== 课后作业 3：合并多个 CSV =====
import csv

# 1. 读取原数据
with open("data/cities_climate.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    all_cities = list(reader)

fieldnames = list(all_cities[0].keys())    # 原列名

# 2. 拆分为南北
north = []
south = []
for c in all_cities:
    if float(c["latitude"]) >= 30:
        north.append(c)
    else:
        south.append(c)

# 3. 写两个文件
with open("output_north.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(north)

with open("output_south.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(south)

print(f"北方城市：{len(north)} 个")
print(f"南方城市：{len(south)} 个")

# 4. 合并到 output_merged.csv，加 region 列
merged_fieldnames = fieldnames + ["region"]
with open("output_merged.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=merged_fieldnames)
    writer.writeheader()
    
    for c in north:
        new_row = dict(c)
        new_row["region"] = "North"
        writer.writerow(new_row)
    
    for c in south:
        new_row = dict(c)
        new_row["region"] = "South"
        writer.writerow(new_row)

print("output_merged.csv 已生成")